# One heterogeneous Darcy problem across CPU, MPI and GPU

This notebook changes **execution**, while preserving the operator, material,
source, fine meshes, approximation spaces and normal-flux interface.
Follow the same workflow as the method tutorials: **meshes → local equations →
global equations → assemble → solve → fields and errors**.

We use a small smooth heterogeneous material so physical pressure and Darcy-flux
errors can be measured against an independently defined analytical solution.
This notebook is an execution/correctness guide, not a timing campaign.

The native CPU/MPI cells require DOLFINx and PETSc/MUMPS, supplied by the locked
`introduction` environment. Set `PYMHM_GUIDE_GPU=1` and use `introduction-gpu`
to execute the actual one/two-GPU cells. GPU availability is stated explicitly;
the ordinary CPU path does not imply GPU assembly.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/ac656ca2f1562b158e3ed70b801130c232079a37983af2cbaff0d75b3dd85f50/heterogeneous_execution-companion.zip"
COMPANION_SHA256 = "ac656ca2f1562b158e3ed70b801130c232079a37983af2cbaff0d75b3dd85f50"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/heterogeneous_execution.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ.get("PYMHM_WORKSPACE", Path.cwd())).resolve()
if not (ROOT / "examples").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "examples").is_dir()), ROOT)
if not (ROOT / "examples").is_dir():
    raise RuntimeError("Extract this notebook's verified companion and set PYMHM_WORKSPACE to it.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
OUTPUT = ROOT / "build" / "guide-execution"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)


## 1. Physical data and independent manufactured source

On the unit square, prescribe zero pressure on the complete exterior boundary.

$$
\begin{aligned}
K(x,y)&=\exp(\sin(8\pi x)\sin(8\pi y)),\\
p_{\mathrm{exact}}(x,y)&=\sin(\pi x)\sin(\pi y),\\
q_{\mathrm{exact}}&=-K\nabla p_{\mathrm{exact}},\\
f&=2\pi^2 Kp_{\mathrm{exact}}-\nabla K\cdot\nabla p_{\mathrm{exact}}.
\end{aligned}
$$

The source comes from independent analytic derivatives, rather than applying a
discrete matrix to a sampled field. The material period is 0.25 and contrast
is $e^2$. The physical Darcy flux is a vector, with both components included
in its error. The primal reconstructed flux is broken; it is not H(div).


In [ ]:
import inspect
from functools import partial
import numpy as np
from IPython.display import Code, display
from examples.introduction.scaling_forms import (
    PeriodicDarcyData, define_ufl_local_equations, exact_pressure, exact_gradient,
)

data = PeriodicDarcyData(period=0.25)
display(Code(inspect.getsource(PeriodicDarcyData), language="python"))


## 2. Build macro and local meshes, then the interface

Four macrorectangles share their geometric faces. Each local mesh has 8×8 fine
quadrilaterals. Its pressure is Q1. Each complete macroface has two continuous
P1 normal-flux coordinates. The normal binding handles face numbering and
outward/global-normal orientation; we do not implement another orientation map.

The importable mesh callable makes process execution portable: the native mesh
and function space are created in the owning worker.


In [ ]:
from pymhm import Equation, ExecutionConfig, MeshHierarchy, assemble, bind_interface, bind_problem
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.meshes.cartesian import CartesianMacroMesh
from examples.guides.heterogeneous_execution import LocalMeshes, measure_fields

macro = CartesianMacroMesh(2, 2)
trace = SkeletonSpace(
    macro, tuple(FaceSpace.uniform(1, 1, continuous=True) for _ in macro.faces)
)
hierarchy = MeshHierarchy(macro, LocalMeshes(macro, refinement=8))
interface = bind_interface(trace, convention="normal")


## 3. Read the executed local UFL equations

The following cell displays the **actual importable mathematical callback**.
It is user-written code, not a problem-solving convenience function.
Its equations are

$$
\begin{aligned}
(K\nabla p_K,\nabla v)_K+\langle\lambda_K,v\rangle_{\partial K}&=(f,v)_K,\\
C_Kp_K&=\langle p_K,\mu_K\rangle_{\partial K}.
\end{aligned}
$$

Read each expression next to its UFL line: `a` is the energy, `L` the source,
`b` the local normal-flux action and `c` the global pressure-continuity pairing.
`kernel` retains the constant and `moments=columns(v * dx)` selects its physical
integral complement. The positive `c` convention produces zero pressure jumps
and zero exterior pressure moments. Native resources stay inside each context.

To change the operator, edit this callback in an importable module. Its UFL
expressions then run unchanged in serial, threads, spawn workers and MPI.


In [ ]:
display(Code(inspect.getsource(define_ufl_local_equations), language="python"))
provider = partial(define_ufl_local_equations, data=data)


## 4. Define the global equation, assemble, then solve

The local `c` equations assemble the pressure continuity and exterior pressure
moments. There is no additional global operator or nonzero pressure load.
`retained=1` allocates one constant coordinate per macrocell. Dirichlet pressure
data fix the physical global pressure gauge.


In [ ]:
problem = bind_problem(
    hierarchy, interface, provider,
    global_equation=Equation(0, 0), retained=1,
)
serial_system = assemble(problem, execution=ExecutionConfig("serial", native_threads=1))
serial_solution = serial_system.solve()
errors_squared = measure_fields(
    problem, hierarchy.items, serial_solution.fields, serial_system.local_metadata
)
serial_errors = {name: float(np.sqrt(value)) for name, value in errors_squared.items()}
print(serial_errors)
print("Original-equation residual:", serial_solution.raw_residual)


## 5. Compare the same coefficients in two CPU execution modes

Only the scheduling policy changes. Workers form the local Schur contributions;
the coordinator sums shared-face entries in deterministic cell order. Native
threads are bounded to avoid nested oversubscription. The rolling window keeps
local jobs in flight without a full batch barrier.

The provider and mesh factory are importable, which is required for spawn on
Windows and Unix alike. No live UFL/DOLFINx object is serialized.


In [ ]:
cpu_errors = {"serial": serial_errors}
for backend in ("thread", "process"):
    system = assemble(
        problem,
        execution=ExecutionConfig(
            backend, workers=2, batch_size=4, pipeline=True, native_threads=1
        ),
    )
    solution = system.solve()
    for reference, current in zip(serial_solution.fields, solution.fields, strict=True):
        np.testing.assert_allclose(reference, current, rtol=1e-10, atol=1e-12)
    sums = measure_fields(problem, hierarchy.items, solution.fields, system.local_metadata)
    cpu_errors[backend] = {name: float(np.sqrt(value)) for name, value in sums.items()}
cpu_errors


## Classical baseline on independently refined conforming meshes

Before using a numerical reference, verify its own refinement against the
available analytical solution. The same UFL callback supplies the volume
operator and source on global 32×32, 64×64 and 128×128 conforming Q1 meshes.
Strong zero pressure replaces the MHM skeletal coupling; no local condensation
is performed. `MultiscaleProblem.from_global` uses the same assemble/solve API.

The next cell displays the actual reference construction, then prints its
pressure and physical Darcy-flux errors. Each reference mesh is finer than the
MHM's 16×16 total fine grid. This is an accuracy baseline, not a matched-space
or timing comparison.

In [ ]:
from examples.guides.heterogeneous_execution import classical_reference
from examples.introduction.scalar import observed_rates

display(Code(inspect.getsource(classical_reference), language="python"))
reference_rows = []
for n in (32, 64, 128):
    reference_evaluator, errors = classical_reference(n)
    reference_rows.append({"n": n, "pressure_L2": errors["pressure_L2"], "flux_L2": errors["flux_L2"]})
print(reference_rows)
for name in ("pressure_L2", "flux_L2"):
    print(name, "observed rates:", observed_rates(
        [1 / row["n"] for row in reference_rows], [row[name] for row in reference_rows]
    ))

## 6. Plot permeability, pressure and physical Darcy flux

Each panel retains the actual macro mesh. Independent incident values are
sampled separately at interfaces instead of averaging reconstructed fields.
The analytical and numerical panels use the same pressure/flux color limits.


In [ ]:
import matplotlib.pyplot as plt
from examples.introduction.scalar import evaluate_qk, rectangle_panel

def numerical_field(points):
    """Evaluate the declared Cartesian Q1 field in its owning macrocell."""
    owners = np.minimum(np.floor(points / macro.spacing).astype(int), [1, 1])
    cell_ids = owners[:, 1] * macro.nx + owners[:, 0]
    pressure = np.empty(len(points))
    gradient = np.empty((len(points), 2))
    for cell in np.unique(cell_ids):
        selected = cell_ids == cell
        fine = hierarchy.local_mesh(int(cell))
        scatter = serial_system.local_metadata[cell]["native_to_cartesian"]
        values = serial_solution.fields[cell][np.argsort(scatter)]
        pressure[selected], gradient[selected] = evaluate_qk(fine, 1, values, points[selected])
    return pressure, gradient

exact_field = lambda x: (exact_pressure(x), exact_gradient(x))
figure, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
settings = [
    ("Permeability K", exact_field, "permeability_xx", None),
    ("Analytical pressure", exact_field, "pressure", (0, 1)),
    ("MHM pressure", numerical_field, "pressure", (0, 1)),
    ("Pressure absolute error", lambda x: (abs(numerical_field(x)[0] - exact_pressure(x)), np.zeros((len(x),2))), "pressure", None),
    ("Analytical Darcy flux magnitude", exact_field, "flux_magnitude", None),
    ("MHM Darcy flux magnitude", numerical_field, "flux_magnitude", None),
]
flux_max = max(
    rectangle_panel(macro, evaluator, quantity="flux_magnitude", permeability=data.permeability)[2].max()
    for evaluator in (exact_field, numerical_field)
)
for ax, (title, evaluator, quantity, limits) in zip(axes.flat, settings, strict=True):
    points, triangles, values = rectangle_panel(
        macro, evaluator, quantity=quantity, permeability=data.permeability, points_per_side=41
    )
    if quantity == "flux_magnitude": limits = (0, flux_max)
    kwargs = {} if limits is None else {"vmin": limits[0], "vmax": limits[1]}
    image = ax.tripcolor(points[:,0], points[:,1], triangles, values, shading="gouraud", **kwargs)
    for location in (0, 0.5, 1):
        ax.axvline(location, color="black", alpha=0.35, linewidth=0.6)
        ax.axhline(location, color="black", alpha=0.35, linewidth=0.6)
    ax.set(title=title, xlabel="x", ylabel="y", aspect="equal")
    figure.colorbar(image, ax=ax, pad=0.025, shrink=0.85)
figure.savefig(OUTPUT / "heterogeneous-fields.png", dpi=180)
plt.show()


In [ ]:
reference_figure, reference_axes = plt.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)
for ax, quantity, title in zip(
    reference_axes, ("pressure", "flux_magnitude"),
    ("Fine conforming Q1 pressure (128×128)", "Fine conforming Darcy flux magnitude"), strict=True,
):
    points, triangles, values = rectangle_panel(
        macro, reference_evaluator, quantity=quantity, permeability=data.permeability, points_per_side=65
    )
    limits = (0, 1) if quantity == "pressure" else (0, flux_max)
    image = ax.tripcolor(points[:,0], points[:,1], triangles, values, shading="gouraud",
                        vmin=limits[0], vmax=limits[1])
    for location in (0, 0.5, 1):
        ax.axvline(location, color="black", alpha=0.35, linewidth=0.6)
        ax.axhline(location, color="black", alpha=0.35, linewidth=0.6)
    ax.set(title=title, xlabel="x", ylabel="y", aspect="equal")
    reference_figure.colorbar(image, ax=ax, pad=0.025, shrink=0.85)
reference_figure.savefig(OUTPUT / "classical-fields.png", dpi=180)
plt.show()

## 7. Distributed MPI on exactly the same local/global problem

Each rank owns a disjoint strided subset of macrocells. The global trace
numbering is common; retained numbering is rank ordered. PETSc distributes
matrix rows and MUMPS solves the global saddle system. The zero exterior
pressure load needs no replicated additive boundary vector.

The helper below only orchestrates the public collective solver and sums
rank-owned squared physical errors. Its actual source is shown. The notebook
launches one and two ranks and compares their field norms to the serial run.


In [ ]:
import json
import subprocess
from examples.guides.heterogeneous_execution import run_mpi

display(Code(inspect.getsource(run_mpi), language="python"))
mpi_command = """import json
from mpi4py import MPI
from examples.guides.heterogeneous_execution import declared_problem, run_mpi
solution, report = run_mpi(declared_problem(), MPI.COMM_WORLD)
if MPI.COMM_WORLD.rank == 0: print(json.dumps(report))
"""
mpi_errors = {}
for ranks in (1, 2):
    result = subprocess.run(
        ["mpiexec", "-n", str(ranks), sys.executable, "-c", mpi_command],
        check=True, capture_output=True, text=True, cwd=ROOT,
    )
    report = json.loads(result.stdout.strip().splitlines()[-1])
    for name, value in serial_errors.items():
        np.testing.assert_allclose(report[name], value, rtol=1e-10, atol=1e-12)
    mpi_errors[ranks] = report
mpi_errors


## 8. CUDA: CPU form assembly, GPU local algebra

This section is explicitly opt-in (`PYMHM_GUIDE_GPU=1`). Its one-device path
uses sparse cuDSS for the constrained saddle systems. Its two-device path
batches small augmented systems into pivoted LU groups, then assembles the
same CPU global system. This example has zero direct global `D/g` blocks and
zero pressure load; a general explicit-record application must retain them.

An SPD-only AMG method cannot be applied directly to an indefinite local
Neumann saddle system. A projected coercive complement requires its own
verified formulation. GPU assembly, factorization and AMG are distinct
capabilities.

The documented native controls execute one-device cuDSS, two-device batched LU
and two MPI ranks each owning a distinct GPU, with coefficient and field
agreement. A four-macrocell example is not a speedup benchmark.


In [ ]:
RUN_GPU = os.environ.get("PYMHM_GUIDE_GPU", "0") == "1"
if RUN_GPU:
    import cupy as cp
    from pymhm import SolverConfig
    from pymhm.core.equations import compile_local_equations
    from pymhm.core.system import HybridSystem
    from pymhm.execution.cuda import condense_multi_gpu

    if cp.cuda.runtime.getDeviceCount() < 2:
        raise RuntimeError("The two-device guide requires two visible CUDA devices.")
    gpu_system = assemble(problem, solvers=SolverConfig(local_solver="cudss"))
    gpu_solution = gpu_system.solve()
    records = [compile_local_equations(problem.local_provider(cell)) for cell in hierarchy.items]
    responses = condense_multi_gpu(
        (record.problem for record in records), devices=(0, 1), batch_size=2, solver="batched"
    )
    batched_system = HybridSystem.from_responses(
        responses, metadata=(record.metadata for record in records)
    )
    batched_solution = batched_system.solve()
    for result in (gpu_solution, batched_solution):
        for reference, current in zip(serial_solution.fields, result.fields, strict=True):
            np.testing.assert_allclose(reference, current, rtol=1e-10, atol=1e-12)
    print("One-device sparse LU and two-device batched LU match the CPU coefficients.")
    gpu_mpi_command = """import json
import cupy as cp
from mpi4py import MPI
from examples.guides.heterogeneous_execution import declared_problem, run_mpi
comm = MPI.COMM_WORLD
node = comm.Split_type(MPI.COMM_TYPE_SHARED)
device = node.rank
node.Free()
with cp.cuda.Device(device):
    solution, report = run_mpi(declared_problem(), comm, local_solver="cudss")
if comm.rank == 0: print(json.dumps(report))
"""
    result = subprocess.run(
        ["mpiexec", "-n", "2", sys.executable, "-c", gpu_mpi_command],
        check=True, capture_output=True, text=True, cwd=ROOT,
    )
    gpu_mpi_report = json.loads(result.stdout.strip().splitlines()[-1])
    for name, value in serial_errors.items():
        np.testing.assert_allclose(gpu_mpi_report[name], value, rtol=1e-10, atol=1e-12)
    print("Two-rank/two-device sparse LU:", gpu_mpi_report)
else:
    print("CUDA section not selected; use introduction-gpu and PYMHM_GUIDE_GPU=1.")


## 9. What this experiment establishes

The physical problem and discrete spaces are fixed. CPU scheduling, distributed
assembly and CUDA local algebra agree within the declared comparison budgets.
The original equations retain their normal verification tolerances. Timings
include different first-use/cache states and are not reported as speedups.

For strong/weak scaling, use the larger reproducible 3D campaigns in the
[Gallery](https://ipes-lncc.github.io/pymhm/gallery/darcy/). A classical conforming reference uses
its own approximation space and must be assessed at stated physical accuracy.

## References

- Harder, Paredes and Valentin (2013), *A family of Multiscale Hybrid-Mixed
  finite element methods for the Darcy equation with rough coefficients*.
  https://doi.org/10.1016/j.jcp.2013.03.019
- Gomes, Pereira, Valentin and Paredes (2017), *On the Implementation of a
  Scalable Simulator for Multiscale Hybrid-Mixed Methods*.
  https://arxiv.org/abs/1703.10435v1
